# AMR-HUB free-running event-class rollout

Input: `/outputs/learnability_analysis/canonical_events.parquet`.

Same whole-shift 80/20 split and count-based policies as the learnability analysis. Each held-out shift is seeded with its first two real event classes; after that each prediction is fed back as the next state, to the end of the shift. Role and roster unit are shift context; the time bucket comes from the held-out event at each position, so event timing is not generated.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random

import pandas as pd
from IPython.display import display

OUT = Path("/outputs/learnability_analysis")
EVENTS_PATH = OUT / "canonical_events.parquet"
ROLLOUT_OUT = OUT / "rollout_inspection"
ROLLOUT_OUT.mkdir(parents=True, exist_ok=True)

TRAIN_FRACTION = 0.80
SEED = 20260813  # as in the learnability script
N_SEED_EVENTS = 2

print("Events:", EVENTS_PATH)
print("Rollout outputs:", ROLLOUT_OUT)
if not EVENTS_PATH.exists():
    raise FileNotFoundError(
        "canonical_events.parquet not found; run the learnability analysis first."
    )

## 1. Load canonical events

In [ ]:
try:
    events = pd.read_parquet(EVENTS_PATH)
except Exception as e:
    print("pandas.read_parquet failed; trying DuckDB fallback:", type(e).__name__)
    import duckdb

    events = duckdb.sql(f"SELECT * FROM read_parquet('{EVENTS_PATH}')").df()

events = events.sort_values(["shift_id", "event_index"]).reset_index(drop=True)

needed = [
    "shift_id",
    "staff_id",
    "event_index",
    "time_bucket",
    "role",
    "roster_unit",
    "interaction_class",
]
missing = [c for c in needed if c not in events.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"Events: {len(events):,}")
print(f"Shifts: {events['shift_id'].nunique():,}")
print(f"Staff: {events['staff_id'].nunique():,}")
display(events[needed].head(15))

## 2. Train/test split

By whole shift. The same staff member can have shifts in both sets.

In [ ]:
shift_ids = sorted(events["shift_id"].dropna().astype(int).unique().tolist())
rng = random.Random(SEED)
rng.shuffle(shift_ids)

cut = max(1, min(len(shift_ids) - 1, int(len(shift_ids) * TRAIN_FRACTION)))
train_ids = set(shift_ids[:cut])
test_ids = set(shift_ids[cut:])

train = events[events["shift_id"].isin(train_ids)].copy()
test = events[events["shift_id"].isin(test_ids)].copy()

print(f"Training shifts: {len(train_ids):,}")
print(f"Test shifts:     {len(test_ids):,}")
print(f"Training events: {len(train):,}")
print(f"Test events:     {len(test):,}")

assert set(train["shift_id"]).isdisjoint(set(test["shift_id"]))

## 3. Training pairs

In [ ]:
def make_class_pairs(df):
    x = df.sort_values(["shift_id", "event_index"]).copy()
    g = x.groupby("shift_id", sort=False)
    x["prev_class"] = g["interaction_class"].shift(1)
    x["action"] = g["interaction_class"].shift(-1)
    x = x[x["action"].notna()].copy()
    x["current_class"] = x["interaction_class"]
    return x[
        [
            "shift_id",
            "staff_id",
            "event_index",
            "current_class",
            "prev_class",
            "action",
            "time_bucket",
            "role",
            "roster_unit",
        ]
    ]


train_pairs = make_class_pairs(train)
test_pairs_one_step = make_class_pairs(test)

print(f"Training pairs: {len(train_pairs):,}")
print(f"One-step test pairs: {len(test_pairs_one_step):,}")
display(train_pairs.head(10))

## 4. Fit policies

In [ ]:
MODEL_STATES = {
    "majority": None,
    "markov_current_class": ["current_class"],
    "current_plus_previous_class": ["current_class", "prev_class"],
    "plus_time_role": ["current_class", "prev_class", "time_bucket", "role"],
    "full_context": [
        "current_class",
        "prev_class",
        "time_bucket",
        "role",
        "roster_unit",
    ],
}

# Fallback from training shifts only.
majority_counts = Counter(train_pairs["action"].astype(str))
majority_action = sorted(majority_counts.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]

print("Training majority next class:", majority_action)
display(
    pd.DataFrame(
        majority_counts.most_common(), columns=["next_class", "training_count"]
    ).assign(training_share=lambda d: d["training_count"] / d["training_count"].sum())
)


def state_key(row, cols):
    vals = []
    for c in cols:
        v = row[c]
        vals.append(None if pd.isna(v) else v)
    return tuple(vals)


def fit_policy(pair_df, state_cols):
    if state_cols is None:
        return None

    counts = defaultdict(Counter)
    for _, r in pair_df.iterrows():
        counts[state_key(r, state_cols)][str(r["action"])] += 1

    policy = {}
    for state, ctr in counts.items():
        policy[state] = sorted(ctr.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
    return policy


policies = {name: fit_policy(train_pairs, cols) for name, cols in MODEL_STATES.items()}

for name, pol in policies.items():
    if pol is None:
        print(f"{name:30s} -> always {majority_action}")
    else:
        print(f"{name:30s} -> {len(pol):,} learned states")

## 5. Rollout

After the two seed events the model only sees its own predictions. For example, with actual `WS → FS → WS → FS` and seed `WS → FS`, a prediction of `FS` for event 3 makes the state for event 4 `FS` (previous `FS`), not `WS`.

In [ ]:
def rollout_shift(shift_df, model_name):
    shift_df = shift_df.sort_values("event_index").reset_index(drop=True)

    actual = shift_df["interaction_class"].astype(str).tolist()
    n = len(actual)

    if n <= N_SEED_EVENTS:
        return None

    generated = actual[:N_SEED_EVENTS].copy()
    state_cols = MODEL_STATES[model_name]
    policy = policies[model_name]

    unseen_states = 0
    predictions = []

    # Event j is predicted from generated events j-1 (current) and j-2 (previous).
    for j in range(N_SEED_EVENTS, n):
        if model_name == "majority":
            pred = majority_action
            state_seen = True
        else:
            context_pos = j - 1

            row = {
                "current_class": generated[j - 1],
                "prev_class": generated[j - 2] if j >= 2 else None,
                "time_bucket": shift_df.loc[context_pos, "time_bucket"],
                "role": shift_df.loc[context_pos, "role"],
                "roster_unit": shift_df.loc[context_pos, "roster_unit"],
            }

            key = state_key(row, state_cols)
            state_seen = key in policy
            pred = policy.get(key, majority_action)
            if not state_seen:
                unseen_states += 1

        generated.append(pred)
        predictions.append(
            {
                "position": j,
                "actual": actual[j],
                "predicted": pred,
                "correct": int(pred == actual[j]),
                "state_seen": int(state_seen),
            }
        )

    pred_df = pd.DataFrame(predictions)

    prefix_correct = 0
    for ok in pred_df["correct"]:
        if ok:
            prefix_correct += 1
        else:
            break

    # Total variation distance between class proportions.
    classes = sorted(set(actual) | set(generated))
    actual_ctr = Counter(actual[N_SEED_EVENTS:])
    gen_ctr = Counter(generated[N_SEED_EVENTS:])
    denom = max(1, n - N_SEED_EVENTS)
    tv = 0.5 * sum(
        abs(actual_ctr.get(c, 0) / denom - gen_ctr.get(c, 0) / denom) for c in classes
    )

    return {
        "shift_id": shift_df.loc[0, "shift_id"],
        "staff_id": shift_df.loc[0, "staff_id"],
        "model": model_name,
        "sequence_length": n,
        "predicted_events": n - N_SEED_EVENTS,
        "rollout_accuracy": float(pred_df["correct"].mean()),
        "exact_generated_suffix_match": int(pred_df["correct"].all()),
        "correct_prefix_after_seed": int(prefix_correct),
        "unseen_state_fraction": (
            unseen_states / max(1, n - N_SEED_EVENTS)
            if model_name != "majority"
            else 0.0
        ),
        "class_composition_TV_distance": float(tv),
        "actual_sequence": " → ".join(actual),
        "generated_sequence": " → ".join(generated),
    }


rollout_rows = []

for shift_id, sdf in test.groupby("shift_id", sort=True):
    for model_name in MODEL_STATES:
        r = rollout_shift(sdf, model_name)
        if r is not None:
            rollout_rows.append(r)

rollouts = pd.DataFrame(rollout_rows)

print(f"Rollout rows: {len(rollouts):,}")
print(f"Test shifts long enough to roll out: {rollouts['shift_id'].nunique():,}")
display(rollouts.head())

## 6. Results

- `rollout_accuracy`: share of generated positions matching the held-out shift.
- `correct_prefix_after_seed`: correct events before the first error.
- `exact_generated_suffix_match`: whole generated suffix correct.
- `class_composition_TV_distance`: difference in class proportions (0 = identical).

In [ ]:
summary = (
    rollouts.groupby("model")
    .agg(
        test_shifts=("shift_id", "nunique"),
        mean_rollout_accuracy=("rollout_accuracy", "mean"),
        median_rollout_accuracy=("rollout_accuracy", "median"),
        exact_suffix_match_rate=("exact_generated_suffix_match", "mean"),
        mean_correct_prefix=("correct_prefix_after_seed", "mean"),
        median_correct_prefix=("correct_prefix_after_seed", "median"),
        mean_unseen_state_fraction=("unseen_state_fraction", "mean"),
        mean_class_composition_TV=("class_composition_TV_distance", "mean"),
    )
    .reset_index()
)

order = list(MODEL_STATES)
summary["model"] = pd.Categorical(summary["model"], categories=order, ordered=True)
summary = summary.sort_values("model")

display(summary)
summary.to_csv(ROLLOUT_OUT / "next_class_rollout_summary.csv", index=False)
rollouts.to_csv(ROLLOUT_OUT / "next_class_rollout_per_shift.csv", index=False)

print("Saved:")
print(ROLLOUT_OUT / "next_class_rollout_summary.csv")
print(ROLLOUT_OUT / "next_class_rollout_per_shift.csv")

## 7. One-step vs rollout accuracy

In [ ]:
one_step_path = OUT / "pooled_learnability.csv"

if one_step_path.exists():
    one_step = pd.read_csv(one_step_path)
    one_step = one_step[one_step["task"].eq("next_class")][["model", "top1"]].copy()
    comp = summary.merge(one_step, on="model", how="left")
    comp = comp.rename(columns={"top1": "one_step_top1"})
    comp["rollout_drop"] = comp["mean_rollout_accuracy"] - comp["one_step_top1"]
    display(
        comp[
            [
                "model",
                "one_step_top1",
                "mean_rollout_accuracy",
                "rollout_drop",
                "mean_correct_prefix",
                "mean_unseen_state_fraction",
            ]
        ]
    )
else:
    print(
        "pooled_learnability.csv not found; rollout results are still available above."
    )

## 8. Inspect a held-out shift

Set `SHIFT_TO_VIEW`. The first two events are the real seed.

In [ ]:
available = sorted(rollouts["shift_id"].unique().tolist())
SHIFT_TO_VIEW = available[0]

view = rollouts[rollouts["shift_id"].eq(SHIFT_TO_VIEW)].copy()

for _, r in view.iterrows():
    print("\n" + "=" * 90)
    print("MODEL:", r["model"])
    print("SHIFT:", r["shift_id"], "| length:", r["sequence_length"])
    print("rollout accuracy:", f"{r['rollout_accuracy']:.1%}")
    print("correct prefix after seed:", r["correct_prefix_after_seed"])
    print()
    print("ACTUAL:")
    print(r["actual_sequence"])
    print()
    print("GENERATED:")
    print(r["generated_sequence"])

## 9. Example shifts

In [ ]:
example_shift_ids = available[:10]
examples = rollouts[rollouts["shift_id"].isin(example_shift_ids)][
    [
        "shift_id",
        "model",
        "sequence_length",
        "rollout_accuracy",
        "correct_prefix_after_seed",
        "actual_sequence",
        "generated_sequence",
    ]
].copy()

display(examples)
examples.to_csv(ROLLOUT_OUT / "next_class_rollout_examples.csv", index=False)
print("Saved:", ROLLOUT_OUT / "next_class_rollout_examples.csv")